# Road Issues — ResNet18 Transfer Learning

Based on the original `CNNmethod(2).ipynb`.

Baseline: SimpleCNN, 224×224 input, 5 classes, test accuracy 85.70%.

This notebook keeps the original dataset workflow and changes the model to a pretrained ResNet18. Experiment 1 freezes the pretrained feature extractor and trains only a new 5-class classifier.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import models
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
zip_path = "/content/drive/MyDrive/Road_Issues.zip"
extract_path = "/content/extracted_dataset"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)


In [ ]:
!pip install -q split-folders


In [ ]:
import splitfolders
data_dir = "/content/extracted_dataset/Road Issues"
split_dir = "/content/split_dataset"
splitfolders.ratio(data_dir, output=split_dir, ratio=(0.70, 0.15, 0.15))
train_dir = "/content/split_dataset/train"
val_dir = "/content/split_dataset/val"
test_dir = "/content/split_dataset/test" 


## Configuration

The original DataLoader uses batch size 32, so this experiment keeps 32. The learning rate is reduced because the pretrained network already contains useful weights.


In [ ]:
batch_size = 32
learning_rate = 0.0001
num_epochs = 10
num_classes = 5


## Transforms

Keep 224×224, but use the normalization expected by the pretrained ImageNet weights.


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])


In [ ]:
train_dataset = ImageFolder(train_dir, transform=train_transform)
val_dataset = ImageFolder(val_dir, transform=transform_val_test)
test_dataset = ImageFolder(test_dir, transform=transform_val_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Classes:", train_dataset.classes)


In [ ]:
images, labels = next(iter(train_loader))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)


## ResNet18 — only the classifier is replaced

All pretrained layers are frozen. The original ResNet18 `fc` layer is replaced with a new layer producing 5 outputs for your five road-issue classes.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

print(model.fc)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)


## Evaluation helper


In [ ]:
def evaluate_loader(loader):
    model.eval()
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(images)

            predictions = outputs.argmax(dim=1)
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    return np.array(all_labels), np.array(all_predictions)


## Training with validation

The best validation checkpoint is saved. The test set is used only for the final evaluation.


In [ ]:
scaler = torch.amp.GradScaler("cuda")
best_val_accuracy = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    val_labels, val_predictions = evaluate_loader(val_loader)
    val_acc = accuracy_score(val_labels, val_predictions) * 100

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {train_loss:.4f} "
        f"Train Accuracy: {train_acc:.2f}% "
        f"Validation Accuracy: {val_acc:.2f}%"
    )

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), "/content/resnet18_best_road_model.pth")
        print("Best model saved.")


## Final test evaluation


In [ ]:
model.load_state_dict(torch.load(
    "/content/resnet18_best_road_model.pth",
    map_location=device,
    weights_only=True
))

test_labels, test_predictions = evaluate_loader(test_loader)

accuracy = accuracy_score(test_labels, test_predictions)
precision = precision_score(test_labels, test_predictions, average="weighted", zero_division=0)
recall = recall_score(test_labels, test_predictions, average="weighted", zero_division=0)
f1 = f1_score(test_labels, test_predictions, average="weighted", zero_division=0)

print(f"Test Accuracy : {accuracy*100:.2f}%")
print(f"Test Precision: {precision*100:.2f}%")
print(f"Test Recall   : {recall*100:.2f}%")
print(f"Test F1-score : {f1*100:.2f}%")


In [ ]:
print(classification_report(
    test_labels,
    test_predictions,
    target_names=train_dataset.classes,
    digits=4,
    zero_division=0
))


In [ ]:
cm = confusion_matrix(test_labels, test_predictions)
print("Confusion Matrix:")
print(cm)


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("ResNet18 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.xticks(range(num_classes), train_dataset.classes, rotation=45, ha="right")
plt.yticks(range(num_classes), train_dataset.classes)

for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
torch.save(model.state_dict(), "/content/resnet18_road_model.pth")
print("Model saved as resnet18_road_model.pth")
